# Vehicle Insurance Fraud Analysis

# Project Context

This notebook documents the data understanding, preparation and exploratory analysis of the Vehicle Insurance Claim Fraud Detection dataset. The analysis focuses on identifying patterns associated with fraudulent and non-fraudulent insurance claims and assessing how the findings can support further investigation and claims prioritisation.

In [1043]:
# Necessary dependencies/libraries

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.naive_bayes import CategoricalNB


## 1. Load Dataset

Import the original Kaggle dataset into Python.

In [1044]:
df = pd.read_csv("insurance_fraud.csv")

## 2. Understand Dataset

Examine the dataset structure, dimensions, attributes, data types and basic distributions.

---

### 2.1 Dataset Overview

In [1045]:
df.head()

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,Age,Fault,PolicyType,VehicleCategory,VehiclePrice,FraudFound_P,PolicyNumber,RepNumber,Deductible,DriverRating,Days_Policy_Accident,Days_Policy_Claim,PastNumberOfClaims,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,21,Policy Holder,Sport - Liability,Sport,more than 69000,0,1,12,300,1,more than 30,more than 30,none,3 years,26 to 30,No,No,External,none,1 year,3 to 4,1994,Liability
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,34,Policy Holder,Sport - Collision,Sport,more than 69000,0,2,15,400,4,more than 30,more than 30,none,6 years,31 to 35,Yes,No,External,none,no change,1 vehicle,1994,Collision
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,47,Policy Holder,Sport - Collision,Sport,more than 69000,0,3,7,400,3,more than 30,more than 30,1,7 years,41 to 50,No,No,External,none,no change,1 vehicle,1994,Collision
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,65,Third Party,Sedan - Liability,Sport,20000 to 29000,0,4,4,400,2,more than 30,more than 30,1,more than 7,51 to 65,Yes,No,External,more than 5,no change,1 vehicle,1994,Liability
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,27,Third Party,Sport - Collision,Sport,more than 69000,0,5,3,400,1,more than 30,more than 30,none,5 years,31 to 35,No,No,External,none,no change,1 vehicle,1994,Collision


---

### 2.2 Dataset dimensions

In [1046]:
df.shape

(15420, 33)

> 15, 420 insurance claim records, with 33 attributes per claim.

---

### 2.3 Dataset attributes

In [1047]:
df.columns

Index(['Month', 'WeekOfMonth', 'DayOfWeek', 'Make', 'AccidentArea',
       'DayOfWeekClaimed', 'MonthClaimed', 'WeekOfMonthClaimed', 'Sex',
       'MaritalStatus', 'Age', 'Fault', 'PolicyType', 'VehicleCategory',
       'VehiclePrice', 'FraudFound_P', 'PolicyNumber', 'RepNumber',
       'Deductible', 'DriverRating', 'Days_Policy_Accident',
       'Days_Policy_Claim', 'PastNumberOfClaims', 'AgeOfVehicle',
       'AgeOfPolicyHolder', 'PoliceReportFiled', 'WitnessPresent', 'AgentType',
       'NumberOfSuppliments', 'AddressChange_Claim', 'NumberOfCars', 'Year',
       'BasePolicy'],
      dtype='str')

---

### 2.4 Dataset structure

In [1048]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15420 entries, 0 to 15419
Data columns (total 33 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Month                 15420 non-null  str  
 1   WeekOfMonth           15420 non-null  int64
 2   DayOfWeek             15420 non-null  str  
 3   Make                  15420 non-null  str  
 4   AccidentArea          15420 non-null  str  
 5   DayOfWeekClaimed      15420 non-null  str  
 6   MonthClaimed          15420 non-null  str  
 7   WeekOfMonthClaimed    15420 non-null  int64
 8   Sex                   15420 non-null  str  
 9   MaritalStatus         15420 non-null  str  
 10  Age                   15420 non-null  int64
 11  Fault                 15420 non-null  str  
 12  PolicyType            15420 non-null  str  
 13  VehicleCategory       15420 non-null  str  
 14  VehiclePrice          15420 non-null  str  
 15  FraudFound_P          15420 non-null  int64
 16  PolicyNumber   

> No missing values: each column shows `15420 non-null`. Data Types: **9** Numerical and **24** String.

---

### 2.5 Inspect Target Variable, `FraudFound_P`

In [1049]:
df["FraudFound_P"].value_counts()

FraudFound_P
0    14497
1      923
Name: count, dtype: int64

In [1050]:
# Target Variable in percentages

df["FraudFound_P"].value_counts(normalize=True) * 100

FraudFound_P
0    94.014267
1     5.985733
Name: proportion, dtype: float64

_The target variable `FraudFound_P` is found to be imbalanced, with 94.01% of claims labelled 0 and 5.99% labelled 1._

## 3. Assess Data Quality

Identify missing values, duplicates, inconsistent values and other potential quality issues.

---

### 3.1 Unique values count for each (24) categorical attributes.

In [1051]:
df.select_dtypes(include="str").nunique().sort_values()

AccidentArea             2
Sex                      2
Fault                    2
PoliceReportFiled        2
WitnessPresent           2
AgentType                2
BasePolicy               3
VehicleCategory          3
Days_Policy_Claim        4
PastNumberOfClaims       4
MaritalStatus            4
NumberOfSuppliments      4
AddressChange_Claim      5
NumberOfCars             5
Days_Policy_Accident     5
VehiclePrice             6
DayOfWeek                7
AgeOfVehicle             8
DayOfWeekClaimed         8
PolicyType               9
AgeOfPolicyHolder        9
Month                   12
MonthClaimed            13
Make                    19
dtype: int64

---

### 3.2 Inspecting the categorical values

In [1052]:
df["Sex"].unique()

<StringArray>
['Female', 'Male']
Length: 2, dtype: str

In [1053]:
for col in ["AccidentArea", "Fault", "PoliceReportFiled", "WitnessPresent", "AgentType"]:
  print(col, ":", df[col].unique(), "\n")

AccidentArea : <StringArray>
['Urban', 'Rural']
Length: 2, dtype: str 

Fault : <StringArray>
['Policy Holder', 'Third Party']
Length: 2, dtype: str 

PoliceReportFiled : <StringArray>
['No', 'Yes']
Length: 2, dtype: str 

WitnessPresent : <StringArray>
['No', 'Yes']
Length: 2, dtype: str 

AgentType : <StringArray>
['External', 'Internal']
Length: 2, dtype: str 



For the 6 binary variables `Sex`, `AccidentArea`, `Fault`, `PoliceReportFiled`, `WitnessPresent` and `AgentType`, we can conclude that:
* There are no spelling/capitalisation inconsistencies.
* The categories are consistent.
* There are no unexpected extra categories.

> Therefore, no need to clean or standardise these columns.

In [1054]:
for col in df.select_dtypes(include="str").columns:
  print(col, ":", df[col].unique(), "\n")

Month : <StringArray>
['Dec', 'Jan', 'Oct', 'Jun', 'Feb', 'Nov', 'Apr', 'Mar', 'Aug', 'Jul', 'May',
 'Sep']
Length: 12, dtype: str 

DayOfWeek : <StringArray>
['Wednesday', 'Friday', 'Saturday', 'Monday', 'Tuesday', 'Sunday', 'Thursday']
Length: 7, dtype: str 

Make : <StringArray>
[    'Honda',    'Toyota',      'Ford',     'Mazda', 'Chevrolet',   'Pontiac',
    'Accura',     'Dodge',   'Mercury',    'Jaguar',    'Nisson',        'VW',
      'Saab',    'Saturn',    'Porche',       'BMW',   'Mecedes',   'Ferrari',
     'Lexus']
Length: 19, dtype: str 

AccidentArea : <StringArray>
['Urban', 'Rural']
Length: 2, dtype: str 

DayOfWeekClaimed : <StringArray>
[  'Tuesday',    'Monday',  'Thursday',    'Friday', 'Wednesday',  'Saturday',
    'Sunday',         '0']
Length: 8, dtype: str 

MonthClaimed : <StringArray>
['Jan', 'Nov', 'Jul', 'Feb', 'Mar', 'Dec', 'Apr', 'Aug', 'May', 'Jun', 'Sep',
 'Oct',   '0']
Length: 13, dtype: str 

Sex : <StringArray>
['Female', 'Male']
Length: 2, dtype: st

Attributes `DayOfWeekClaimed` and `MonthClaimed` have records with the numerical value **0**. We need to investigate their legitimacy and how they affects the analysis.

---

### 3.3 Identifying the 'errorneous' values

In [1055]:
df["DayOfWeekClaimed"].value_counts()

DayOfWeekClaimed
Monday       3757
Tuesday      3375
Wednesday    2951
Thursday     2660
Friday       2497
Saturday      127
Sunday         52
0               1
Name: count, dtype: int64

In [1056]:
df["MonthClaimed"].value_counts()

MonthClaimed
Jan    1446
May    1411
Mar    1348
Oct    1339
Jun    1293
Feb    1287
Nov    1285
Apr    1271
Sep    1242
Jul    1225
Dec    1146
Aug    1126
0         1
Name: count, dtype: int64

In [1057]:
df[(df["DayOfWeekClaimed"] == "0") | (df["MonthClaimed"] == "0")]

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,Age,Fault,PolicyType,VehicleCategory,VehiclePrice,FraudFound_P,PolicyNumber,RepNumber,Deductible,DriverRating,Days_Policy_Accident,Days_Policy_Claim,PastNumberOfClaims,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
1516,Jul,2,Monday,Honda,Rural,0,0,1,Male,Single,0,Policy Holder,Sedan - All Perils,Sedan,more than 69000,0,1517,15,400,2,more than 30,none,none,new,16 to 17,No,No,External,none,no change,1 vehicle,1994,All Perils


We can see that a single record at _index 1516_ has invalid values for both `DayOfWeekClaimed` and `MonthClaimed`. The decision is to keep the records since other attributes do not appear to be corrupted.

---

### 3.4 Inspecting Duplicates

In [1058]:
df.duplicated().sum()

np.int64(0)

> **No** exact duplicate rows were found.

---

### 3.5 Inspecting Numerical Values

In [1059]:
df.describe()

,WeekOfMonth,WeekOfMonthClaimed,Age,FraudFound_P,PolicyNumber,RepNumber,Deductible,DriverRating,Year
count,15420.000000,15420.000000,15420.000000,15420.000000,15420.000000,15420.000000,15420.000000,15420.000000,15420.000000
mean,2.788586,2.693969,39.855707,0.059857,7710.500000,8.483268,407.704280,2.487808,1994.866472
std,1.287585,1.259115,13.492377,0.237230,4451.514911,4.599948,43.950998,1.119453,0.803313
min,1.000000,1.000000,0.000000,0.000000,1.000000,1.000000,300.000000,1.000000,1994.000000
25%,2.000000,2.000000,31.000000,0.000000,3855.750000,5.000000,400.000000,1.000000,1994.000000
50%,3.000000,3.000000,38.000000,0.000000,7710.500000,8.000000,400.000000,2.000000,1995.000000
75%,4.000000,4.000000,48.000000,0.000000,11565.250000,12.000000,400.000000,3.000000,1996.000000
max,5.000000,5.000000,80.000000,1.000000,15420.000000,16.000000,700.000000,4.000000,1996.000000


`PolicyNumber` data suggests that this field is being used as an identifier for records. However, no definite conclusion is made at this stage.

`Age` column has a minimum value of **0**? This suggests further investigation.

In [1060]:
(df["Age"] == 0).sum()

np.int64(320)

**320** records of `Age` have the value **0**!

In [1061]:
df.loc[df["Age"] == 0, "AgeOfPolicyHolder"].value_counts()

AgeOfPolicyHolder
16 to 17    320
Name: count, dtype: int64

It would seem that `Age = 0` belongs to the category of `AgeOfPolicyHolder: 16 to 17`. But why?

In [1062]:
df["Age"].value_counts().sort_index()

Age
0     320
16      9
17      6
18     48
19     32
     ... 
76     42
77     29
78     35
79     20
80     32
Name: count, Length: 66, dtype: int64

> `Age = 0` seems to be unusual encoded value that occurs for each of the 320 records whose `AgeOfPolicyHolder` is 16 to 17. Further investigation may be required.

---

### 3.6 Check for missing values explicitly

In [1063]:
df.isna().sum().sum()

np.int64(0)

Section `2.4` demonstrated that we had **no** missing values and _np.int64(0)_ confirms it.

---

## 4. Clean the Data

We assess the dataset, identify the issues that are genuinely present, and apply preparation only where justified.

In [1064]:
# Work on a clean dataframe.

df_clean = df.copy()

---

### 4.1 Issue 1: Handle the invalid `month` and `day of week` values

In [1065]:
df_clean["DayOfWeekClaimed"] = df_clean["DayOfWeekClaimed"].replace("0", "Unknown")
df_clean["MonthClaimed"] = df_clean["MonthClaimed"].replace("0", "Unknown")

Now, **0** as both an invalid _Day_ and _Month_, has been replaced to `Unknown`.

---

### 4.2 Issue 2.1: Investigate `Age = 0`

In [1066]:
df_clean.groupby("AgeOfPolicyHolder")["Age"].agg(["min", "max", "mean", "count"])

,min,max,mean,count
AgeOfPolicyHolder,,,,
16 to 17,0,0,0.000000,320
18 to 20,16,17,16.400000,15
21 to 25,18,20,18.814815,108
26 to 30,21,25,22.941272,613
31 to 35,26,35,30.548006,5593
36 to 40,36,45,40.483304,4043
41 to 50,46,55,50.423267,2828
51 to 65,56,65,60.441092,1392
over 65,66,80,72.783465,508


We investigated the relationship between `Age` and `AgeOfPolicyHolder` to try understanding what `Age = 0` meant. **However**, we can see that the `AgeOfPolicyHolder` seems to have been shifted upwards and we now need to understand this first.

---

### 4.3 Issue 2.2: Investigate `Age` and `AgeOfPolicyHolder`

We investigate the relationship between `Age` and `AgeOfPolicyHolder` to try understanding what `Age = 0` means.

In [1067]:
pd.crosstab(df_clean["AgeOfPolicyHolder"], df_clean["Age"])

Age,0,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80
AgeOfPolicyHolder,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
16 to 17,320,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
18 to 20,0,9,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
21 to 25,0,0,0,48,32,28,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
26 to 30,0,0,0,0,0,0,127,125,122,135,104,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
31 to 35,0,0,0,0,0,0,0,0,0,0,0,535,540,560,552,596,550,544,574,573,569,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
36 to 40,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,406,410,384,435,383,423,401,404,411,386,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
41 to 50,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,296,308,291,265,290,279,276,253,288,282,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
51 to 65,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,146,144,134,138,156,145,112,136,146,135,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
over 65,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,42,31,32,32,27,40,45,32,35,34,42,29,35,20,32


The above analysis further confirms that `AgeOfPolicyHolder` have been wrongly inputed. Hence, we will create a new column named `AgeGroup` which contains the correct categories based on `Age`. 

The **320 records** with `Age = 0` were treated as having an unknown age. Since there was insufficient evidence to determine their actual age group, they were not assigned to an existing age category. Instead, they were assigned an `Unknown` category.

In [1068]:
bins = [15, 17, 20, 25, 30, 35, 40, 50, 65, 80]

labels = [
  "16 to 17",
  "18 to 20",
  "21 to 25",
  "26 to 30",
  "31 to 35",
  "36 to 40",
  "41 to 50",
  "51 to 65",
  "over 65"
]

df_clean["AgeGroup"] = pd.cut(
  df_clean["Age"],
  bins=bins,
  labels=labels
)

df_clean["AgeGroup"] = df_clean["AgeGroup"].cat.add_categories("Unknown")

df_clean.loc[df_clean["Age"] == 0, "AgeGroup"] = "Unknown"

We now verify the new AgeGroup / Age relationship:

In [1069]:
pd.crosstab(
  df_clean["AgeGroup"],
  df_clean["Age"],
  dropna=False
)

Age,0,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80
AgeGroup,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
16 to 17,0,9,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
18 to 20,0,0,0,48,32,28,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
21 to 25,0,0,0,0,0,0,127,125,122,135,104,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
26 to 30,0,0,0,0,0,0,0,0,0,0,0,535,540,560,552,596,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
31 to 35,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,550,544,574,573,569,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
36 to 40,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,406,410,384,435,383,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
41 to 50,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,423,401,404,411,386,296,308,291,265,290,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
51 to 65,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,279,276,253,288,282,146,144,134,138,156,145,112,136,146,135,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
over 65,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,42,31,32,32,27,40,45,32,35,34,42,29,35,20,32


---

### 4.4 Issue 3: Investigate `PolicyNumber` and `RepNumber`

First, we determine uniqueness of `PolicyNumber`

In [1070]:
df_clean["PolicyNumber"].nunique()

15420

Since we obtained **15420**, we can conclude that every row has a **unique** policy number. Hence the atribute is being used as an **identifier** and will potentially be discarded.

Then, we determine uniqueness of `RepNumber`

In [1071]:
df_clean["RepNumber"].nunique()

16

**16** cannot mean that each record is unique. In fact, it identifies each **insurance representative** using a unique number (1 - 16) and a representative/agent may be associated to **multiple claims**.

---

## 5. Prepare Data Types and Attributes

Convert variables into appropriate data types and distinguish between numerical, categorical and identifier attributes.

In [1072]:
df_clean.dtypes

Month                        str
WeekOfMonth                int64
DayOfWeek                    str
Make                         str
AccidentArea                 str
DayOfWeekClaimed             str
MonthClaimed                 str
WeekOfMonthClaimed         int64
Sex                          str
MaritalStatus                str
Age                        int64
Fault                        str
PolicyType                   str
VehicleCategory              str
VehiclePrice                 str
FraudFound_P               int64
PolicyNumber               int64
RepNumber                  int64
Deductible                 int64
DriverRating               int64
Days_Policy_Accident         str
Days_Policy_Claim            str
PastNumberOfClaims           str
AgeOfVehicle                 str
AgeOfPolicyHolder            str
PoliceReportFiled            str
WitnessPresent               str
AgentType                    str
NumberOfSuppliments          str
AddressChange_Claim          str
NumberOfCa

The attribute types were reviewed and although many are stored as integers or strings in python, their analytical interpretation was considered before deciding whether conversion was necessary.

At this stage, no conversion was deemed necesary. Categorical data were retained and their representation justified. However, depending on the analytical techniques or statistical modelling, appropriate conversions may be required.

The resulting attribute classification is summarised below.

| Attribute type  | Attributes            |
| --------------- | ----------------------------------------------------------------------------------------------------------------------------- |
| **Numerical**   | `Age`, `Deductible`   |
| **Categorical** | `Make`, `AccidentArea`, `Sex`, `MaritalStatus`, `Fault`, `PolicyType`, `VehicleCategory`, `VehiclePrice`, `DriverRating`, `Days_Policy_Accident`, `Days_Policy_Claim`, `PastNumberOfClaims`, `AgeOfVehicle`, `AgeOfPolicyHolder`, `PoliceReportFiled`, `WitnessPresent`, `AgentType`, `NumberOfSuppliments`, `AddressChange_Claim`, `NumberOfCars`, `BasePolicy`, `AgeGroup` |
| **Temporal**    | `Month`, `WeekOfMonth`, `DayOfWeek`, `DayOfWeekClaimed`, `MonthClaimed`, `WeekOfMonthClaimed`, `Year`      |
| **Identifier**  | `PolicyNumber`        |
| **Categorical Identifier** | `RepNumber`|
| **Target**      | `FraudFound_P`        |


---

## 6. Select Relevant Attributes

Identify variables that are useful for the business questions and analytical objectives while removing irrelevant or unsuitable attributes.

**Removed Attributes:**

- `PolicyNumber` since it is an **identifier**.
- `AgeOfPolicyHolder` since the attribute is **inconsistent with the observed `Age` values**.
- `Age` since `AgeGroup` provides a more interpretable categorical representation for analysis and BI visualisation.
- `PolicyType` since it is a **composite attribute** formed from `VehicleCategory` and `BasePolicy`.

In [1073]:
# Work with a clean copy

df_prepared = df_clean.copy()

In [1074]:
# Remove irrelevant/unsitable attributes

df_prepared = df_prepared.drop(
  columns=["PolicyNumber", "AgeOfPolicyHolder", "Age", "PolicyType"]
)

In [1075]:
print(f"\nColumns after selecting relevant attributes: ")

# To show full column head/names
pd.set_option("display.max_columns", None)
print(df_prepared.columns.tolist())

df_prepared.head()


Columns after selecting relevant attributes: 
['Month', 'WeekOfMonth', 'DayOfWeek', 'Make', 'AccidentArea', 'DayOfWeekClaimed', 'MonthClaimed', 'WeekOfMonthClaimed', 'Sex', 'MaritalStatus', 'Fault', 'VehicleCategory', 'VehiclePrice', 'FraudFound_P', 'RepNumber', 'Deductible', 'DriverRating', 'Days_Policy_Accident', 'Days_Policy_Claim', 'PastNumberOfClaims', 'AgeOfVehicle', 'PoliceReportFiled', 'WitnessPresent', 'AgentType', 'NumberOfSuppliments', 'AddressChange_Claim', 'NumberOfCars', 'Year', 'BasePolicy', 'AgeGroup']


,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,Fault,VehicleCategory,VehiclePrice,FraudFound_P,RepNumber,Deductible,DriverRating,Days_Policy_Accident,Days_Policy_Claim,PastNumberOfClaims,AgeOfVehicle,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy,AgeGroup
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,Policy Holder,Sport,more than 69000,0,12,300,1,more than 30,more than 30,none,3 years,No,No,External,none,1 year,3 to 4,1994,Liability,21 to 25
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,Policy Holder,Sport,more than 69000,0,15,400,4,more than 30,more than 30,none,6 years,Yes,No,External,none,no change,1 vehicle,1994,Collision,31 to 35
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,Policy Holder,Sport,more than 69000,0,7,400,3,more than 30,more than 30,1,7 years,No,No,External,none,no change,1 vehicle,1994,Collision,41 to 50
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,Third Party,Sport,20000 to 29000,0,4,400,2,more than 30,more than 30,1,more than 7,Yes,No,External,more than 5,no change,1 vehicle,1994,Liability,51 to 65
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,Third Party,Sport,more than 69000,0,3,400,1,more than 30,more than 30,none,5 years,No,No,External,none,no change,1 vehicle,1994,Collision,26 to 30


---

## 7. Create the Prepared Dataset

Produce a clean and analysis-ready dataset for subsequent analysis, modelling and BI development.

### 7.1 Final Dataset Validation

In [1076]:
print("Shape:", df_prepared.shape)
print("Missing values:", df_prepared.isna().sum().sum())
print("Duplicate rows:", df_prepared.duplicated().sum())
print("Data types:")
print(df_prepared.dtypes.value_counts())

Shape: (15420, 30)
Missing values: 0
Duplicate rows: 0
Data types:
str         22
int64        7
category     1
Name: count, dtype: int64


### 7.2 Verify Target Variable

In [1077]:
df_prepared["FraudFound_P"].value_counts()

FraudFound_P
0    14497
1      923
Name: count, dtype: int64

### 7.3 Save Prepared Dataset

In [1078]:
df_prepared.to_csv("insurance_fraud_prepared.csv", index=False)

import os
print(os.path.exists("insurance_fraud_prepared.csv"))

True


---

## 8. Insight Creation

### 8.1 Exploratory Data Analysis (EDA)

Use the prepared claims data to discover patterns that are relevant to the insurance-fraud business problems.

### 8.1.1 Overall Fraud Baseline

In [1079]:
fraud_counts = df_prepared["FraudFound_P"].value_counts()

fraud_rate = (
    df_prepared["FraudFound_P"].mean() * 100
)

print("Fraudulent claims:", fraud_counts[1])
print("Non-fraudulent claims:", fraud_counts[0])
print(f"Overall fraud rate: {fraud_rate:.2f}%")

Fraudulent claims: 923
Non-fraudulent claims: 14497
Overall fraud rate: 5.99%


---

### 8.1.2 Policyholder Characteristics

In [1080]:
# Fraud rate by sex

fraud_by_sex = (
  df_prepared.groupby("Sex")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_sex["fraud_rate"] = fraud_by_sex["fraud_rate"] * 100

fraud_by_sex

,count,fraudulent,fraud_rate
Sex,,,
Female,2420,105,4.338843
Male,13000,818,6.292308


**Sex:** `Male` claims have a higher observed fraud rate (6.29%) than `female` claims (4.34%).

In [1081]:
# Fraud rate by marital status

fraud_by_marital = (
  df_prepared.groupby("MaritalStatus")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_marital["fraud_rate"] = fraud_by_marital["fraud_rate"] * 100

fraud_by_marital

,count,fraudulent,fraud_rate
MaritalStatus,,,
Divorced,76,3,3.947368
Married,10625,639,6.014118
Single,4684,278,5.935098
Widow,35,3,8.571429


**Marital Status:** `Widow` claims have the highest observed fraud rate (8.57%), while `divorced` claims have the lowest (3.95%); however, both groups contain very few fraudulent claims in numbers alone.

In [1082]:
# Fraud rate by age group

fraud_by_age_group = (
  df_prepared.groupby("AgeGroup", observed=True)["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_age_group["fraud_rate"] = fraud_by_age_group["fraud_rate"] * 100

fraud_by_age_group

,count,fraudulent,fraud_rate
AgeGroup,,,
16 to 17,15,2,13.333333
18 to 20,108,16,14.814815
21 to 25,613,33,5.383361
26 to 30,2783,170,6.108516
31 to 35,2810,190,6.761566
36 to 40,2018,127,6.293360
41 to 50,3475,186,5.352518
51 to 65,2770,138,4.981949
over 65,508,30,5.905512


**Age Group:** Claims from the `18–20` age group have the highest observed fraud rate (14.81%), followed by `16–17` (13.33%), while the `51–65` group has the lowest (4.98%); the younger groups have relatively small claim counts, even if the rates are high.

In [1083]:
# Fraud rate by driver rating

fraud_by_driver_rating = (
  df_prepared.groupby("DriverRating")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_driver_rating["fraud_rate"] = fraud_by_driver_rating["fraud_rate"] * 100

fraud_by_driver_rating

,count,fraudulent,fraud_rate
DriverRating,,,
1,3944,232,5.882353
2,3801,214,5.630097
3,3884,242,6.230690
4,3791,235,6.198892


**Driver Rating:** Fraud rates are **relatively similar** across all driver ratings, ranging from 5.63% (Rating 2) to 6.23% (Rating 3), suggesting **little variation** in observed fraud rate by driver rating.

---

### 8.1.3 Vehicle Characteristics

In [1084]:
# Fraud rate by make

fraud_by_make = (
  df_prepared.groupby("Make")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_make["fraud_rate"] = fraud_by_make["fraud_rate"] * 100

fraud_by_make

,count,fraudulent,fraud_rate
Make,,,
Accura,472,59,12.500000
BMW,15,1,6.666667
Chevrolet,1681,94,5.591910
Dodge,109,2,1.834862
Ferrari,2,0,0.000000
Ford,450,33,7.333333
Honda,2801,179,6.390575
Jaguar,6,0,0.000000
Lexus,1,0,0.000000


**Make:** Fraud rates vary substantially across vehicle makes, with `Accura` at 12.50% among the larger groups, while  makes like `Mecedes`, `Saab` and `Saturn`, with very few claims show extreme rates that should be interpreted cautiously.

In [1085]:
# Fraud rate by vehicle category

fraud_by_vehicle_category = (
  df_prepared.groupby("VehicleCategory")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_vehicle_category["fraud_rate"] = (
  fraud_by_vehicle_category["fraud_rate"] * 100
)

fraud_by_vehicle_category

,count,fraudulent,fraud_rate
VehicleCategory,,,
Sedan,9671,795,8.220453
Sport,5358,84,1.567749
Utility,391,44,11.253197


**Vehicle Category:** `Utility` vehicles have the highest observed fraud rate (11.25%), followed by `Sedans` (8.22%), while `Sport` vehicles have a much lower rate (1.57%).

In [1086]:
# Fraud rate by vehicle price

fraud_by_vehicle_price = (
  df_prepared.groupby("VehiclePrice")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_vehicle_price["fraud_rate"] = (
  fraud_by_vehicle_price["fraud_rate"] * 100
)

fraud_by_vehicle_price

,count,fraudulent,fraud_rate
VehiclePrice,,,
20000 to 29000,8079,421,5.211041
30000 to 39000,3533,175,4.953297
40000 to 59000,461,31,6.724512
60000 to 69000,87,4,4.597701
less than 20000,1096,103,9.397810
more than 69000,2164,189,8.733826


**Vehicle Price:** Claims for vehicles priced `less than 20,000` have a relatively high fraud rate (9.40%), while those priced `more than 69,000` also show a high rate (8.73%).

In [1087]:
# Fraud rate by vehicle age

fraud_by_vehicle_age = (
  df_prepared.groupby("AgeOfVehicle")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_vehicle_age["fraud_rate"] = (
  fraud_by_vehicle_age["fraud_rate"] * 100
)

fraud_by_vehicle_age

,count,fraudulent,fraud_rate
AgeOfVehicle,,,
2 years,73,3,4.109589
3 years,152,13,8.552632
4 years,229,21,9.170306
5 years,1357,95,7.000737
6 years,3448,228,6.612529
7 years,5807,325,5.596694
more than 7,3981,206,5.174579
new,373,32,8.579088


**Vehicle Age:** Fraud rates **vary across vehicle ages**, with `4-year-old` vehicles having the highest rate (9.17%) among the established age groups, while vehicles `more than 7` years old have a lower rate (5.17%).

---

### 8.1.4 Accident / Claim Characteristics

In [1088]:
# Fraud rate by accident area

fraud_by_area = (
  df_prepared.groupby("AccidentArea")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_area["fraud_rate"] = fraud_by_area["fraud_rate"] * 100

fraud_by_area

,count,fraudulent,fraud_rate
AccidentArea,,,
Rural,1598,133,8.322904
Urban,13822,790,5.715526


**Accident Area: Rural** claims have a higher observed fraud rate (8.32%) than **urban** claims (5.72%).

In [1089]:
# Fraud rate by fault

fraud_by_fault = (
  df_prepared.groupby("Fault")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_fault["fraud_rate"] = fraud_by_fault["fraud_rate"] * 100

fraud_by_fault

,count,fraudulent,fraud_rate
Fault,,,
Policy Holder,11230,886,7.889581
Third Party,4190,37,0.883055


**Fault:** Claims where the `policy holder was at fault` have a substantially higher observed fraud rate (7.89%) than claims involving a `third party` (0.88%).

In [1090]:
# Fraud rate by police report filed

fraud_by_police_report = (
  df_prepared.groupby("PoliceReportFiled")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_police_report["fraud_rate"] = (
  fraud_by_police_report["fraud_rate"] * 100
)

fraud_by_police_report

,count,fraudulent,fraud_rate
PoliceReportFiled,,,
No,14992,907,6.049893
Yes,428,16,3.738318


**Police Report Filed:** Claims `without a police report` have a higher observed fraud rate (6.05%) than claims `with a police report` (3.74%).

In [1091]:
# Fraud rate by witness present

fraud_by_witness = (
    df_prepared.groupby("WitnessPresent")["FraudFound_P"]
    .agg(
        count="count",
        fraudulent="sum",
        fraud_rate="mean"
    )
)

fraud_by_witness["fraud_rate"] = (
    fraud_by_witness["fraud_rate"] * 100
)

fraud_by_witness

,count,fraudulent,fraud_rate
WitnessPresent,,,
No,15333,920,6.000130
Yes,87,3,3.448276


**Witness Present:** Claims `without a witness` have a higher observed fraud rate (6.00%) than claims `with a witness` (3.45%); *however*, the witness-present group contains only 87 claims.

---

### 8.1.5 Policy / Claim Characteristics

In [1092]:
# Fraud rate by days from policy to accident

fraud_by_policy_accident = (
  df_prepared.groupby("Days_Policy_Accident")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_policy_accident["fraud_rate"] = (
  fraud_by_policy_accident["fraud_rate"] * 100
)

fraud_by_policy_accident

,count,fraudulent,fraud_rate
Days_Policy_Accident,,,
1 to 7,14,1,7.142857
15 to 30,49,3,6.122449
8 to 15,55,5,9.090909
more than 30,15247,905,5.935594
none,55,9,16.363636


**Days from Policy to Accident:** Claims occurring with `no recorded policy-to-accident` interval have the highest observed fraud rate (16.36%), while claims with `more than 30 days` have a rate of 5.94%; the **none** group is relatively small though.

In [1093]:
# Fraud rate by days from policy to claim

fraud_by_policy_claim = (
  df_prepared.groupby("Days_Policy_Claim")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_policy_claim["fraud_rate"] = (
  fraud_by_policy_claim["fraud_rate"] * 100
)

fraud_by_policy_claim

,count,fraudulent,fraud_rate
Days_Policy_Claim,,,
15 to 30,56,6,10.714286
8 to 15,21,3,14.285714
more than 30,15342,914,5.957502
none,1,0,0.000000


**Days from Policy to Claim:** Claims with `8–15 days` between policy and claim have the highest observed fraud rate (14.29%), followed by `15–30` days (10.71%); *however*, these groups are very small.

In [1094]:
# Fraud rate by past number of claims

fraud_by_past_claims = (
  df_prepared.groupby("PastNumberOfClaims")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_past_claims["fraud_rate"] = (
  fraud_by_past_claims["fraud_rate"] * 100
)

fraud_by_past_claims

,count,fraudulent,fraud_rate
PastNumberOfClaims,,,
1,3573,222,6.213266
2 to 4,5485,294,5.360073
more than 4,2010,68,3.383085
none,4352,339,7.789522


**Past Number of Claims:** Claims with `no previous claims` have the highest observed fraud rate (7.79%), while claims with `more than 4` previous claims have the lowest (3.38%).

In [1095]:
# Fraud rate by number of supplements

fraud_by_supplements = (
  df_prepared.groupby("NumberOfSuppliments")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_supplements["fraud_rate"] = (
  fraud_by_supplements["fraud_rate"] * 100
)

fraud_by_supplements

,count,fraudulent,fraud_rate
NumberOfSuppliments,,,
1 to 2,2489,159,6.388108
3 to 5,2017,97,4.809122
more than 5,3867,195,5.042669
none,7047,472,6.697886


**Number of Supplements:** Fraud rates are **relatively similar across supplement categories**, ranging from 4.81% for `3–5 supplements` to 6.70% for claims with `no supplements`.

In [1096]:
# Fraud rate by address change

fraud_by_address_change = (
  df_prepared.groupby("AddressChange_Claim")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_address_change["fraud_rate"] = (
  fraud_by_address_change["fraud_rate"] * 100
)

fraud_by_address_change

,count,fraudulent,fraud_rate
AddressChange_Claim,,,
1 year,170,11,6.470588
2 to 3 years,291,51,17.525773
4 to 8 years,631,33,5.229794
no change,14324,825,5.759564
under 6 months,4,3,75.000000


**Address Change:** Claims with an address change `2–3 years` ago have an observed fraud rate of 17.53%, while claims with `no change` have a rate of 5.76%; the `under 6 months` category has a very high rate (75%) *but* contains only 4 claims.

In [1097]:
# Fraud rate by base policy

fraud_by_base_policy = (
  df_prepared.groupby("BasePolicy")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_base_policy["fraud_rate"] = (
  fraud_by_base_policy["fraud_rate"] * 100
)

fraud_by_base_policy

,count,fraudulent,fraud_rate
BasePolicy,,,
All Perils,4449,452,10.159586
Collision,5962,435,7.296209
Liability,5009,36,0.718706


**Base Policy:** `All Perils` policies have the highest observed fraud rate (10.16%), followed by `Collision` (7.30%), while `Liability` policies have a much lower rate (0.72%).

---

### 8.1.6 Temporal Patterns

In [1098]:
# Fraud rate by accident day of week

fraud_by_day = (
  df_prepared.groupby("DayOfWeek")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_day["fraud_rate"] = (
  fraud_by_day["fraud_rate"] * 100
)

fraud_by_day

,count,fraudulent,fraud_rate
DayOfWeek,,,
Friday,2445,154,6.298569
Monday,2616,160,6.116208
Saturday,1982,132,6.659939
Sunday,1745,122,6.991404
Thursday,2173,120,5.522319
Tuesday,2300,120,5.217391
Wednesday,2159,115,5.326540


**Accident Day of Week:** Observed fraud rates **vary modestly** across accident days, ranging from `5.22%` on `Tuesday` to `6.99%` on `Sunday`.

In [1099]:
# Fraud rate by accident week of month

fraud_by_week = (
  df_prepared.groupby("WeekOfMonth")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_week["fraud_rate"] = (
  fraud_by_week["fraud_rate"] * 100
)

fraud_by_week

,count,fraudulent,fraud_rate
WeekOfMonth,,,
1,3187,200,6.275494
2,3558,225,6.323777
3,3640,215,5.906593
4,3398,192,5.650383
5,1637,91,5.558949


**Accident Week of Month:** Observed fraud rates are **relatively similar** across the five weeks of the month, ranging from `5.56%` in `Week 5` to `6.32%` in `Week 2`.

In [1100]:
# Fraud rate by accident month

fraud_by_month = (
  df_prepared.groupby("Month")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_month["fraud_rate"] = (
  fraud_by_month["fraud_rate"] * 100
)

fraud_by_month

,count,fraudulent,fraud_rate
Month,,,
Apr,1280,80,6.250000
Aug,1127,84,7.453416
Dec,1285,62,4.824903
Feb,1266,82,6.477093
Jan,1411,87,6.165840
Jul,1257,60,4.773270
Jun,1321,80,6.056018
Mar,1360,102,7.500000
May,1367,94,6.876372


**Accident Month:** Observed fraud rates **vary** across accident months, with `March` having the highest rate (`7.50%`) and `November` the lowest (`3.83%`).

In [1101]:
# Fraud rate by year

fraud_by_year = (
  df_prepared.groupby("Year")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_year["fraud_rate"] = (
  fraud_by_year["fraud_rate"] * 100
)

fraud_by_year

,count,fraudulent,fraud_rate
Year,,,
1994,6142,409,6.659069
1995,5195,301,5.794033
1996,4083,213,5.216752


**Year:** The observed fraud rate decreases across the recorded years, from `6.66%` in `1994` to `5.22%` in `1996`.

In [1102]:
# Fraud rate by claim day of week

fraud_by_claim_day = (
  df_prepared.groupby("DayOfWeekClaimed")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_claim_day["fraud_rate"] = (
  fraud_by_claim_day["fraud_rate"] * 100
)

fraud_by_claim_day

,count,fraudulent,fraud_rate
DayOfWeekClaimed,,,
Friday,2497,164,6.567881
Monday,3757,216,5.749268
Saturday,127,10,7.874016
Sunday,52,3,5.769231
Thursday,2660,144,5.413534
Tuesday,3375,198,5.866667
Unknown,1,0,0.000000
Wednesday,2951,188,6.370722


**Claim Day of Week:** Observed fraud rates **vary** across claim days, with `Saturday` having the highest rate (`7.87%`) and `Thursday` the lowest (`5.41%`); *however*, Saturday claims are relatively few.

In [1103]:
# Fraud rate by claim week of month

fraud_by_claim_week = (
  df_prepared.groupby("WeekOfMonthClaimed")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_claim_week["fraud_rate"] = (
  fraud_by_claim_week["fraud_rate"] * 100
)

fraud_by_claim_week

,count,fraudulent,fraud_rate
WeekOfMonthClaimed,,,
1,3450,220,6.376812
2,3720,208,5.591398
3,3583,221,6.168016
4,3433,209,6.087970
5,1234,65,5.267423


**Claim Week of Month:** Observed fraud rates are **relatively similar** across claim weeks, ranging from `5.27%` in `Week 5` to `6.38%` in `Week 1`.

In [1104]:
# Fraud rate by claim month

fraud_by_claim_month = (
  df_prepared.groupby("MonthClaimed")["FraudFound_P"]
  .agg(
    count="count",
    fraudulent="sum",
    fraud_rate="mean"
  )
)

fraud_by_claim_month["fraud_rate"] = (
  fraud_by_claim_month["fraud_rate"] * 100
)

fraud_by_claim_month

,count,fraudulent,fraud_rate
MonthClaimed,,,
Apr,1271,82,6.451613
Aug,1126,92,8.170515
Dec,1146,49,4.275742
Feb,1287,78,6.060606
Jan,1446,92,6.362379
Jul,1225,56,4.571429
Jun,1293,78,6.032483
Mar,1348,97,7.195846
May,1411,102,7.228916


**Claim Month:** Observed fraud rates **vary** across claim months, with `August` having the highest rate (`8.17%`) and `November` the lowest (`3.58%`).

---

### 8.2 Decision Tree Classification and Bagging

A Decision Tree classifier was developed to investigate whether claim characteristics can distinguish fraudulent from non-fraudulent claims. Bagging was subsequently applied by combining multiple Decision Tree classifiers into an ensemble for comparison with the baseline model.

In [1105]:
# Create a working copy for Decision Tree analysis

df_dt = df_prepared.copy()

df_dt.shape

(15420, 30)

In [1106]:
# Adjusting the DT by excluding attributes that are not useful as predictors

attributes_to_remove = [
  "Month",
  "WeekOfMonth",
  "DayOfWeek",
  "DayOfWeekClaimed",
  "MonthClaimed",
  "WeekOfMonthClaimed",
  "Year",
  "RepNumber"
]

In [1107]:
# Separate target (FraudFound_P) from predictor attributes

X = df_dt.drop(
  columns=["FraudFound_P"] + attributes_to_remove
)

y = df_dt["FraudFound_P"]

In [1108]:
# Identify categorical and numerical predictors
categorical_features = X.select_dtypes(
  include=["object", "category", "str"]
).columns.tolist()

numerical_features = X.select_dtypes(
  include=["number"]
).columns.tolist()

print("Categorical features:", len(categorical_features))
print(categorical_features)

print("\nNumerical features:", len(numerical_features))
print(numerical_features)

Categorical features: 19
['Make', 'AccidentArea', 'Sex', 'MaritalStatus', 'Fault', 'VehicleCategory', 'VehiclePrice', 'Days_Policy_Accident', 'Days_Policy_Claim', 'PastNumberOfClaims', 'AgeOfVehicle', 'PoliceReportFiled', 'WitnessPresent', 'AgentType', 'NumberOfSuppliments', 'AddressChange_Claim', 'NumberOfCars', 'BasePolicy', 'AgeGroup']

Numerical features: 2
['Deductible', 'DriverRating']


In [1109]:
# Internal One-Hot encoder for categorical attributes

preprocessor = ColumnTransformer (
  transformers = [
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("numerical", "passthrough", numerical_features)
  ]
)

**8.2.1 Decision Tree: Baseline Model**

In [1110]:
# Decision Tree formation

dt_pipeline = Pipeline (
    steps = [
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(random_state=42)) # Reproductible analysis
    ]
)

dt_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numerical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float

In [1111]:
# Stratified Split due to imbalanced Target (5.99% fraud vs 94.01% non-fraud)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape) # 80% of records used as 'Training Set'
print("Testing set:", X_test.shape)   # 20% of records used as 'Testing Set'

Training set: (12336, 21)
Testing set: (3084, 21)


In [1112]:
# Train the Decision Tree

dt_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](21,)","['Make','AccidentArea','Sex',...,'NumberOfCars','BasePolicy','AgeGroup']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,21
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numerical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='

In [1113]:
# Test the Decision Tree

y_pred = dt_pipeline.predict(X_test)

print("Number of predictions:", len(y_pred))
print("\nPredicted class distribution:")
print(pd.Series(y_pred).value_counts())

Number of predictions: 3084

Predicted class distribution:
0    2873
1     211
Name: count, dtype: int64


In [1114]:
# Evaluate the Decision Tree

cm_dt = confusion_matrix(y_test, y_pred)

print(cm_dt)

print()

# Calculate Decision Tree evaluation metrics

TN, FP, FN, TP = cm_dt.ravel() # Order the CM

accuracy_dt = (TP + TN) / (TP + TN + FP + FN)
error_rate_dt = (FP + FN) / (TP + TN + FP + FN)
sensitivity_dt = TP / (TP + FN)
specificity_dt = TN / (TN + FP)

print(f"Accuracy: {accuracy_dt:.4f} ({accuracy_dt * 100:.2f}%)")
print(f"Error Rate: {error_rate_dt:.4f} ({error_rate_dt * 100:.2f}%)")
print(f"Sensitivity: {sensitivity_dt:.4f} ({sensitivity_dt * 100:.2f}%)")
print(f"Specificity: {specificity_dt:.4f} ({specificity_dt * 100:.2f}%)")

[[2719  180]
 [ 154   31]]

Accuracy: 0.8917 (89.17%)
Error Rate: 0.1083 (10.83%)
Sensitivity: 0.1676 (16.76%)
Specificity: 0.9379 (93.79%)


In [1115]:
# Understanding the Decision Tree

dt_model = dt_pipeline.named_steps["classifier"]

print("Number of classes:", dt_model.n_classes_) # How do we classify Fraud_Found_P?
print("Number of input features:", dt_model.n_features_in_)
print("Number of leaves:", dt_model.get_n_leaves())

Number of classes: 2
Number of input features: 94
Number of leaves: 1380


In [1116]:
# Retrieve the feature names used internally by the Decision Tree

feature_names = dt_pipeline.named_steps["preprocessor"].get_feature_names_out()

print("Number of feature names:", len(feature_names))
print(feature_names)

print()

# Inspect feature importance

feature_importance = pd.DataFrame({
  "Feature": feature_names,
  "Importance": dt_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
  by="Importance",
  ascending=False
)

feature_importance.head(15)

print()

# Get human-readable view of the DT (top level only)

tree_rules = export_text(
  dt_model,
  feature_names=list(feature_names),
  max_depth=3 # Can re-adjust to view all
)

print(tree_rules)

Number of feature names: 94
['categorical__Make_Accura' 'categorical__Make_BMW'
 'categorical__Make_Chevrolet' 'categorical__Make_Dodge'
 'categorical__Make_Ferrari' 'categorical__Make_Ford'
 'categorical__Make_Honda' 'categorical__Make_Jaguar'
 'categorical__Make_Lexus' 'categorical__Make_Mazda'
 'categorical__Make_Mecedes' 'categorical__Make_Mercury'
 'categorical__Make_Nisson' 'categorical__Make_Pontiac'
 'categorical__Make_Porche' 'categorical__Make_Saab'
 'categorical__Make_Saturn' 'categorical__Make_Toyota'
 'categorical__Make_VW' 'categorical__AccidentArea_Rural'
 'categorical__AccidentArea_Urban' 'categorical__Sex_Female'
 'categorical__Sex_Male' 'categorical__MaritalStatus_Divorced'
 'categorical__MaritalStatus_Married' 'categorical__MaritalStatus_Single'
 'categorical__MaritalStatus_Widow' 'categorical__Fault_Policy Holder'
 'categorical__Fault_Third Party' 'categorical__VehicleCategory_Sedan'
 'categorical__VehicleCategory_Sport'
 'categorical__VehicleCategory_Utility'
 'cat

In [1117]:
# Providing the DT Model with unseen data

new_claim = pd.DataFrame([{
  "Make": "Chevrolet",
  "AccidentArea": "Rural",
  "Sex": "Male",
  "MaritalStatus": "Married",
  "Fault": "Policy Holder",
  "VehicleCategory": "Sedan",
  "VehiclePrice": "less than 20000",
  "Deductible": 400,
  "DriverRating": 4,
  "Days_Policy_Accident": "none",
  "Days_Policy_Claim": "8 to 15",
  "PastNumberOfClaims": "none",
  "AgeOfVehicle": "4 years",
  "PoliceReportFiled": "No",
  "WitnessPresent": "No",
  "AgentType": "External",
  "NumberOfSuppliments": "none",
  "AddressChange_Claim": "2 to 3 years",
  "NumberOfCars": "1 vehicle",
  "BasePolicy": "All Perils",
  "AgeGroup": "18 to 20"
}])

# Own prediction by the model

prediction = dt_pipeline.predict(new_claim)

if prediction[0] == 1:
  print("Prediction: Fraud")
else:
  print("Prediction: Non-Fraud")

Prediction: Non-Fraud


**8.2.2 Bagging: Multiple Decision Trees**

In [1118]:
# Bagging using Decision Trees as the base classifier

bagging_pipeline = Pipeline(
  steps=[
    ("preprocessor", preprocessor),
    (
      "classifier",
      BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=10, # Create 10 DT classifiers within the Bagging ensemble.
        random_state=42
      )
    )
  ]
)

In [1119]:
# Train the Bagging model

bagging_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](21,)","['Make','AccidentArea','Sex',...,'NumberOfCars','BasePolicy','AgeGroup']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,21
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numerical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='

In [1120]:
# Test the Bagging model

y_pred_bagging = bagging_pipeline.predict(X_test)

print("Number of predictions:", len(y_pred_bagging))

print("\nPredicted class distribution:")
print(pd.Series(y_pred_bagging).value_counts())

Number of predictions: 3084

Predicted class distribution:
0    3041
1      43
Name: count, dtype: int64


In [1121]:
# Evaluate the Bagging model

cm_bagging = confusion_matrix(y_test, y_pred_bagging)

print(cm_bagging)

print()

TN, FP, FN, TP = cm_bagging.ravel()

accuracy_bagging = (TP + TN) / (TP + TN + FP + FN)
error_rate_bagging = (FP + FN) / (TP + TN + FP + FN)
sensitivity_bagging = TP / (TP + FN)
specificity_bagging = TN / (TN + FP)

print(f"Accuracy: {accuracy_bagging:.4f} ({accuracy_bagging * 100:.2f}%)")
print(f"Error Rate: {error_rate_bagging:.4f} ({error_rate_bagging * 100:.2f}%)")
print(f"Sensitivity: {sensitivity_bagging:.4f} ({sensitivity_bagging * 100:.2f}%)")
print(f"Specificity: {specificity_bagging:.4f} ({specificity_bagging * 100:.2f}%)")

[[2866   33]
 [ 175   10]]

Accuracy: 0.9326 (93.26%)
Error Rate: 0.0674 (6.74%)
Sensitivity: 0.0541 (5.41%)
Specificity: 0.9886 (98.86%)


**8.2.3 Comparison of Models**

The performance of the baseline Decision Tree and the Bagging ensemble was compared using the same testing data and evaluation measures. This allows the effect of combining multiple Decision Trees to be examined against the single-tree baseline.

In [1122]:
# Compare Decision Tree and Bagging performance

model_comparison = pd.DataFrame({
  "Model": ["Decision Tree", "Bagging"],
  "Accuracy": [accuracy_dt, accuracy_bagging],
  "Error Rate": [error_rate_dt, error_rate_bagging],
  "Sensitivity": [sensitivity_dt, sensitivity_bagging],
  "Specificity": [specificity_dt, specificity_bagging]
})

model_comparison

,Model,Accuracy,Error Rate,Sensitivity,Specificity
0,Decision Tree,0.891699,0.108301,0.167568,0.937910
1,Bagging,0.932555,0.067445,0.054054,0.988617


---

### 8.3 Naïve Bayes Classifier

In [1123]:
# Create a working copy for Bayesian Classification

df_nb = df_prepared.copy()

df_nb.shape

(15420, 30)

In [1124]:
# Adjusting the NB Model by using 'categorical' attributes only, as predictors

nb_attributes = [
  "Make",
  "AccidentArea",
  "Sex",
  "MaritalStatus",
  "Fault",
  "VehicleCategory",
  "VehiclePrice",
  "Days_Policy_Accident",
  "Days_Policy_Claim",
  "PastNumberOfClaims",
  "AgeOfVehicle",
  "PoliceReportFiled",
  "WitnessPresent",
  "AgentType",
  "NumberOfSuppliments",
  "AddressChange_Claim",
  "NumberOfCars",
  "BasePolicy",
  "AgeGroup"
]

X_nb = df_nb[nb_attributes]
y_nb = df_nb["FraudFound_P"]

In [1125]:
# Preparing the categorical attributes for the NB Classification

encoder_nb = OrdinalEncoder(
  handle_unknown="use_encoded_value",
  unknown_value=-1
)

X_nb_encoded = encoder_nb.fit_transform(X_nb)

In [1126]:
# Create the training and testing split

X_nb_train, X_nb_test, y_nb_train, y_nb_test = train_test_split(
  X_nb,
  y_nb,
  test_size=0.20,
  random_state=42,
  stratify=y_nb
)

print("Training predictors:", X_nb_train.shape)
print("Testing predictors:", X_nb_test.shape)

Training predictors: (12336, 19)
Testing predictors: (3084, 19)


In [1127]:
# Encode categorical attributes

encoder_nb = OrdinalEncoder(
  handle_unknown="use_encoded_value",
  unknown_value=-1
)

X_nb_train_encoded = encoder_nb.fit_transform(X_nb_train) # Training data
X_nb_test_encoded = encoder_nb.transform(X_nb_test)       # Testing data

In [1128]:
# Train the NB Classifier

nb_model = CategoricalNB()

nb_model.fit(X_nb_train_encoded, y_nb_train)

,"alpha alpha: float, default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"min_categories min_categories: int or array-like of shape (n_features,), default=NoneMinimum number of categories per feature.- integer: Sets the minimum number of categories per feature to `n_categories` for each features.- array-like: shape (n_features,) where `n_categories[i]` holds the minimum number of categories for the ith column of the input.- None (default): Determines the number of categories automatically from the training data... versionadded:: 0.24",None
Name,Type,Value
"category_count_ category_count_: list of arrays of shape (n_features,)Holds arrays of shape (n_classes, n_categories of respective feature)for each feature. Each array provides the number of samplesencountered for each class and category of the specific feature.",list,"[array([[3.250... 4.000e+00]]), array([[ 1211...01., 637.]]), array([[1835.... 84., 654.]]), array([[6.200..., 2.000e+00]]), ...]"
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[11598., 738.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.06,-2.82]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_log_prob_ feature_log_prob_: list of arrays of shape (n_features,)Holds arrays of shape (n_classes, n_categories of respective feature)for each feature. Each array provides the empirical log probabilityof categories given the respective feature and class, ``P(x_i|y)``.",list,"[array([[-3.57...-5.01992534]]), array([[-2.25...-0.1483119 ]]), array([[-1.84...-0.12201495]]), array([[-5.21...-5.51073695]]), ...]"


In [1129]:
# Make predictions on the test data

y_nb_pred = nb_model.predict(X_nb_test_encoded)

print("Number of predictions:", len(y_nb_pred))
print("Predicted class distribution:")
print(pd.Series(y_nb_pred).value_counts())

Number of predictions: 3084
Predicted class distribution:
0    3044
1      40
Name: count, dtype: int64


In [1130]:
nb_probabilities = nb_model.predict_proba(X_nb_test_encoded)

nb_results = pd.DataFrame({
  "Actual": y_nb_test.values,
  "Predicted": y_nb_pred,
  "P(Non-Fraud)": nb_probabilities[:, 0],
  "P(Fraud)": nb_probabilities[:, 1]
})

nb_results.head(10)

,Actual,Predicted,P(Non-Fraud),P(Fraud)
0,0,0,0.997524,0.002476
1,0,0,0.755735,0.244265
2,0,0,0.883282,0.116718
3,0,0,0.905800,0.094200
4,0,0,0.997548,0.002452
5,0,0,0.932689,0.067311
6,0,0,0.986699,0.013301
7,1,0,0.878893,0.121107
8,0,0,0.959087,0.040913
9,0,0,0.972244,0.027756


In [1131]:
# Evaluate NB using Confusion Matrix

cm_nb = confusion_matrix(y_nb_test, y_nb_pred)
print(cm_nb)

# CM Metrics

TN, FP, FN, TP = cm_nb.ravel()

total = TN + FP + FN + TP
P = TP + FN
N = TN + FP

accuracy_nb = (TP + TN) / total
error_rate_nb = (FP + FN) / total
sensitivity_nb = TP / P
specificity_nb = TN / N

print(f"Accuracy: {accuracy_nb:.4f} ({accuracy_nb * 100:.2f}%)")
print(f"Error Rate: {error_rate_nb:.4f} ({error_rate_nb * 100:.2f}%)")
print(f"Sensitivity: {sensitivity_nb:.4f} ({sensitivity_nb * 100:.2f}%)")
print(f"Specificity: {specificity_nb:.4f} ({specificity_nb * 100:.2f}%)")

[[2868   31]
 [ 176    9]]
Accuracy: 0.9329 (93.29%)
Error Rate: 0.0671 (6.71%)
Sensitivity: 0.0486 (4.86%)
Specificity: 0.9893 (98.93%)


In [1132]:
# Create a completely unseen claim for Naïve Bayes

new_claim_nb = pd.DataFrame([{
  "Make": "Chevrolet",
  "AccidentArea": "Rural",
  "Sex": "Male",
  "MaritalStatus": "Married",
  "Fault": "Policy Holder",
  "VehicleCategory": "Sedan",
  "VehiclePrice": "less than 20000",
  "Days_Policy_Accident": "none",
  "Days_Policy_Claim": "8 to 15",
  "PastNumberOfClaims": "none",
  "AgeOfVehicle": "4 years",
  "PoliceReportFiled": "No",
  "WitnessPresent": "No",
  "AgentType": "External",
  "NumberOfSuppliments": "none",
  "AddressChange_Claim": "2 to 3 years",
  "NumberOfCars": "1 vehicle",
  "BasePolicy": "All Perils",
  "AgeGroup": "18 to 20"
}])

# Encode the completely unseen claim

new_claim_nb_encoded = encoder_nb.transform(new_claim_nb)

# Predict the class of the unseen claim

prediction_nb = nb_model.predict(new_claim_nb_encoded)
probabilities_nb = nb_model.predict_proba(new_claim_nb_encoded)

print("NB Model Prediction:", "Fraud" if prediction_nb[0] == 1 else "Non-Fraud")
print()
print(f"Non-Fraud probability: {probabilities_nb[0][0] * 100:.2f}%")
print(f"Fraud probability: {probabilities_nb[0][1] * 100:.2f}%")


NB Model Prediction: Fraud

Non-Fraud probability: 0.94%
Fraud probability: 99.06%


---